# Step 2 — Rotulagem, treino e classificação da série temporal

Exporta rótulos de referência (ESA WorldCover + malha viária OSM), treina um Random Forest e uma rede neural densa, classifica toda a série temporal e calcula % de área por classe/ano.

A lógica reutilizável vive em [`src/classification.py`](../src/classification.py) (rotulagem, treino, classificação, plots) e [`src/indices.py`](../src/indices.py) (`load_features`, com NDVI/NDWI/NDBI + EVI/SAVI/BSI/MNDWI/IBI/NDMI) — este notebook só chama essas funções. Para rodar via linha de comando, veja `scripts/step2_classificacao_imagens.py`.

In [ ]:
import os
import sys
from pathlib import Path

# Garante que a raiz do repositório está no sys.path e é o cwd (ver step1).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'requirements.txt').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import ee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

from src.classification import (
    CLASS_NAMES, LABELS_DIR, RAW_DIR,
    build_label_raster, classify_image, compute_class_percentages,
    export_worldcover_labels, extract_training_samples, get_road_mask,
    load_metadata, plot_mask_overlay, plot_timeseries, remap_worldcover, tif_path,
    train_neural_network, train_random_forest,
)
from src.indices import load_features

load_dotenv()  # carrega variáveis de ambiente do arquivo .env, se existir

In [ ]:
# Autenticação e parâmetros
# ee.Authenticate()

EE_PROJECT = os.environ.get('EE_PROJECT')
if not EE_PROJECT:
    raise RuntimeError(
        "Defina a variável de ambiente EE_PROJECT com o ID do seu projeto no Google Cloud "
        "antes de rodar esta célula (ex.: PowerShell: $env:EE_PROJECT = 'seu-projeto-id'; "
        "ou crie um arquivo .env com EE_PROJECT=seu-projeto-id)."
    )

ee.Initialize(project=EE_PROJECT)

out_dir = RAW_DIR
processed_dir = 'data/processed'
name_datacenter = 'Ascenty_Vinhedo'
reference_year = 2024

In [ ]:
# Ler o metadata.json da extração (step1)
meta = load_metadata(name_datacenter, out_dir)
band_names = meta['bands']
ref_tif = tif_path(name_datacenter, reference_year, out_dir)
print(meta)

In [ ]:
# Rótulos de referência: ESA WorldCover
label_path = export_worldcover_labels(meta, LABELS_DIR)
print(label_path)

In [ ]:
# Remapear WorldCover para as 5 classes do projeto
remapped = remap_worldcover(label_path)
print(remapped.shape, np.unique(remapped, return_counts=True))

In [ ]:
# Máscara de estradas (OpenStreetMap)
road_mask = get_road_mask(ref_tif, buffer_m=4)
print('Pixels de estrada:', road_mask.sum())

In [ ]:
# Combinar labels (WorldCover + estradas)
labels = build_label_raster(remapped, road_mask)
print(np.unique(labels, return_counts=True))

In [ ]:
# Carregar features (bandas + índices espectrais) do ano de referência
feature_stack, nodata_mask = load_features(ref_tif, band_names)
print('Shape das features:', feature_stack.shape)

In [ ]:
# Amostragem de pixels de treino
X, y = extract_training_samples(feature_stack, labels, nodata_mask)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

In [ ]:
# Treinar Random Forest
rf_model = train_random_forest(X_train, y_train, X_test, y_test)

In [ ]:
# Treinar Rede Neural
nn_model, scaler = train_neural_network(X_train, y_train, X_test, y_test)

In [ ]:
# Classificar toda a série temporal
rows = []
for year in meta['year_list']:
    path = tif_path(name_datacenter, year, out_dir)
    if not os.path.exists(path):
        print(f'[{year}] arquivo não encontrado, pulei.')
        continue

    fstack, nmask = load_features(path, band_names)
    classified_rf = classify_image(fstack, nmask, rf_model)
    classified_nn = classify_image(fstack, nmask, nn_model, scaler)

    pct_rf = compute_class_percentages(classified_rf)
    pct_nn = compute_class_percentages(classified_nn)

    for classe in CLASS_NAMES:
        rows.append({'ano': year, 'classe': classe, 'modelo': 'Random Forest',
                      'percentual': pct_rf[classe]['percentual'], 'area_km2': pct_rf[classe]['area_km2']})
        rows.append({'ano': year, 'classe': classe, 'modelo': 'Rede Neural',
                      'percentual': pct_nn[classe]['percentual'], 'area_km2': pct_nn[classe]['area_km2']})
    print(f'[{year}] classificado.')

df = pd.DataFrame(rows)
os.makedirs(processed_dir, exist_ok=True)
df.to_csv(os.path.join(processed_dir, f'{name_datacenter}_cobertura_por_ano.csv'), index=False)
df.head(10)

In [ ]:
# Gráfico da evolução ao longo dos anos
plot_timeseries(df, name_datacenter)

# Salvar máscaras (overlay RGB + classificação) por ano

In [ ]:
os.makedirs('imagens_jpg', exist_ok=True)

for year in meta['year_list']:
    path = tif_path(name_datacenter, year, out_dir)
    if not os.path.exists(path):
        print(f'[{year}] arquivo não encontrado, pulei.')
        continue

    fstack, nmask = load_features(path, band_names)
    classified_rf = classify_image(fstack, nmask, rf_model)

    plot_mask_overlay(
        fstack,
        classified_rf,
        titulo=f'{name_datacenter} {year} - Random Forest',
        alpha=0.5,
        salvar_em=f'imagens_jpg/{name_datacenter}_{year}_overlay.jpg',
        mostrar=False,
    )

print('Todas as máscaras geradas.')